# Compartment calling in *A. rouxii*

Compartment calling using `cis_eigs` from cooltools following tutorial found at https://cooltools.readthedocs.io/en/latest/notebooks/compartments_and_saddles.html. Gene density is used as phasing track.

## Import packages 

In [ ]:
#Standard packages
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import os, subprocess

In [ ]:
#Packages for handeling cooler files 
import cooler
import cooltools
import bioframe

In [ ]:
#Packages for plotting the heatmap
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MaxNLocator
import cooltools.lib.plotting

In [ ]:
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.4'):
    raise AssertionError("tutorials rely on cooltools version 0.5.4 or higher,"+
                         "please check your cooltools version and update to the latest")

## Set format for file

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

## Importing mcool, gene density and repeat density files

In [ ]:
# Load BED file
gene_density_AmyRoux = pd.read_csv('/Users/emma/Documents/NMBU/Master/Master/Data/Gene annotation/gene_density_AmyRoux_80000.bed', 
                        sep='\t', header=None, names=['chrom','start','end', 'gene_count'])

In [ ]:
# Load BED file
repeat_density_AmyRoux = pd.read_csv('/Users/emma/Documents/NMBU/Master/Master/Data/repeats_density_AmyRoux_80000.bed', 
                        sep='\t', header=None, names=['chrom','start','end', 'repeat_count'])

In [ ]:
# To print which resolutions are stored in the mcool, use list_coolers
cooler.fileops.list_coolers('/Users/emma/Documents/NMBU/Master/Master/Koding_python/ggAmyRoux1.mcool')

In [ ]:
# Load the data at resolution 80 000
AmyRoux_file = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/Koding_python/ggAmyRoux1.mcool::resolutions/80000') 

# The resolution of mcool file
resolution = AmyRoux_file.binsize
print(resolution)

In [ ]:
# Inspect gene density file
display(gene_density_AmyRoux) # Display the genome coverage table

## Import centromeres predicted using *M.lusitanicus* as a reference

In [ ]:
centromeres = {
    "subgenom1_SUPER_1": (3011, 62819),
    "subgenom1_SUPER_2": (3357007, 3370569),
    "subgenom1_SUPER_10": (72483, 88619),
    "subgenom1_SUPER_3": (981857, 998763),
    "subgenom1_SUPER_4": (2697696, 2699846),
    "subgenom1_SUPER_5": (3509447, 3535106),
    "subgenom1_SUPER_6": (3359543, 3382640),
    "subgenom1_SUPER_7": (2461517, 2920085),
    "subgenom1_SUPER_8": (1904802, 1905129),
    "subgenom2_SUPER_2": (3364087, 3410879),
}

## Filtering

In [ ]:
# Filtering out scaffolds 
min_size = 100_000

# Keep only chromosomes/scaffolds that are large enough
chroms_to_keep = [c for c in AmyRoux_file.chromnames if AmyRoux_file.chromsizes[c] >= min_size and "Scaffold" not in c]

print("Chromosomes kept:", chroms_to_keep) 

## Calculating per-chromosome compartmentalization using cooltools

In [ ]:
# Get the bins from the cooler file
bins = AmyRoux_file.bins()[:] 
display(bins)

In [ ]:
# Define the genomic regions used for analysis 
view_df = pd.DataFrame({'chrom': AmyRoux_file.chromnames,
                        'start': 0,
                        'end': AmyRoux_file.chromsizes.values,
                        'name': AmyRoux_file.chromnames}
                      )
display(view_df)

In [ ]:
# Eigenedecompostion using gene density as phasing track. 
cis_eigs = cooltools.eigs_cis(
                        AmyRoux_file,
                        gene_density_AmyRoux, # Orient using gene density
                        view_df=view_df, 
                        n_eigs=3, # Obtain the first 3 eigenvectors
                        )

# cis_eigs[0] returns eigenvalues, here we focus on eigenvectors
eigenvector_track_all = cis_eigs[1][['chrom','start','end','E1', 'E2', 'E3']] # I added E2 and E3 because I wanted to see all the eigenvectors
display(eigenvector_track_all.head())

eigenvector_track = cis_eigs[1][['chrom','start','end','E1']] # the code from cooltools
display(eigenvector_track.head())

### Look at correlation with eigenvectors 

In [ ]:
# Merge eigenvector and gene density
merged = pd.merge(eigenvector_track_all, gene_density_AmyRoux, on=['chrom','start','end'])
# Drop rows with any NaN values in E1 or gene_count
merged_clean = merged.dropna(subset=['E1','E2','E3','gene_count'])

# Compute correlation on the cleaned dataframe
correlation = np.corrcoef(merged_clean['E1'], merged_clean['gene_count'])[0,1]
print(f"Correlation between E1 and gene content: {correlation:.2f}")
correlation = np.corrcoef(merged_clean['E2'], merged_clean['gene_count'])[0,1]
print(f"Correlation between E2 and gene content: {correlation:.2f}")
correlation = np.corrcoef(merged_clean['E3'], merged_clean['gene_count'])[0,1]
print(f"Correlation between E3 and gene content: {correlation:.2f}")

In [ ]:
# Save the eigenvectors in a tsv file
#eigenvector_track_all.to_csv("eigenvectors_AmyRoux80kb_GD.tsv", sep='\t', index=False)
# Used this for generating the gene density boxplots 

## Plotting compartments calling

### Plott chromosomes with predicted centromeres

In [ ]:
# Loop through chromosomes with predicted centromere 

for chrom, (centromere_start, centromere_end) in centromeres.items():
    
    # Extract chromosome number
    chromosome_number = chrom.split('_')[-1]
    print(f"Plotting chromosome {chromosome_number} ({chrom})")

    # Convert centromere coordinates to bins
    centromere_start_bin = centromere_start // resolution
    centromere_end_bin = centromere_end // resolution

    # Hi-C matrix for given chromosome
    matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
    n_bins = matrix.shape[0]

    # Eigenvectors for given chromosome
    evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
    E1 = evec_chr['E1'].values
    E2 = evec_chr['E2'].values
    E3 = evec_chr['E3'].values

    # Gene density values for given chromosome
    gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
    gene_density = gene_density_chr["gene_count"].values

    # Repeat density values for given chromosome
    repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
    repeat_density = repeat_density_chr["repeat_count"].values
    
    # Plotting 
    ## Plot Hi-C and tracks
    fig, ax = plt.subplots(figsize=(10, 10))

    im = ax.matshow(
        matrix,
        norm=LogNorm(vmax=0.1), # log
        cmap='bwr'
    )
    ax.set_xlim(0, n_bins)
    ax.set_ylim(n_bins, 0)

    ## Colorbar showing corrected frequencies
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label('Corrected frequencies', fontsize=12)
    cbar.ax.tick_params(labelsize=10)

    
    ## Genomic x-axis in Mb
    genomic_ticks = np.arange(0, n_bins, step=10)
    genomic_labels = (genomic_ticks * resolution) / 1e6
    ax.set_yticks(genomic_ticks)
    ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
    ax.set_ylabel("Genomic Position (Mb)", fontsize=12)


    ## Predicted centromere shown as dashed lines in green in Hi-C matrix
    ax.axhline(centromere_start_bin, color='green', linestyle=':')
    ax.axvline(centromere_start_bin, color='green', linestyle=':')
    ax.axhline(centromere_end_bin, color='green', linestyle=':')
    ax.axvline(centromere_end_bin, color='green', linestyle=':')
    ax.legend(['Centromere'], loc='upper right')
    
    ax_prev = ax # Make the plots share x-axis

    # E1 Compartment track (bars)
    ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
    positions = np.arange(len(E1))
    ### Colors for A/B compartments
    colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")

    ax1.bar(positions, 
            E1, 
            width=1.0, 
            color=colors, 
            edgecolor="none")
    
    ax1.axhline(0, color='black', lw=0.5)
    ax1.set_ylabel('E1', fontsize=12)
    ax1.set_xticks([])
    ax_prev = ax1

    ## Compartment boundaries (E1 sign changes)
    boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]
    for b in boundaries:
        ax.axhline(b, color='k', lw=0.5)
        ax.axvline(b, color='k', lw=0.5)

    ## E2 track
    ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    positions_E2 = np.arange(len(E2))
    ### Colors for A/B compartments
    colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")

    ax2.bar(positions_E2, 
            E2, 
            width=1.0, 
            color=colors_E2, 
            edgecolor="none")
    
    ax2.axhline(0, color='black', lw=0.5)
    ax2.set_ylabel('E2', fontsize=12)
    ax2.set_xticks([])
    ax_prev = ax2

    ## E3 track
    ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    positions_E3 = np.arange(len(E3))
    ### Colors for A/B compartments
    colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
    ax3.bar(positions_E3, 
            E3, 
            width=1.0, 
            color=colors_E3, 
            edgecolor="none")
    
    ax3.axhline(0, color='black', lw=0.5)
    ax3.set_ylabel('E3', fontsize=12)
    ax3.set_xticks([])
    ax_prev = ax3

    ## Gene density line plot
    ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
    ax_gene.set_ylabel("Gene\nDensity", fontsize=12)
    ax_gene.set_xticks([])
    ax_gene.set_ylim(0, max(gene_density)*1.1)
    ax_prev = ax_gene

    ## Repeat density line plot
    ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
    ax_repeat.set_ylabel("Repeat\nDensity", fontsize=12)
    ax_repeat.set_xticks([])
    ax_repeat.set_ylim(0, max(repeat_density)*1.1)


    # Title for plot
    fig.text(
        0.5, 0.91,
        f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
        ha="center",
        va="top",
        fontsize=20,
        color='#0E7C7BFF',
        fontname='Times New Roman'
    )

    plt.show()


### Plott all filtered chromosomes (including chromosomes without predicted centromeres)

In [ ]:
# Loop through filtered chromosomes
for chrom in chroms_to_keep:
    print(f"Plotting chromosome: {chrom}")
    
    # Extract chromosome number
    chromosome_number = chrom.split('_')[-1]
    print(f"Plotting chromosome {chromosome_number} ({chrom})")
    
    # Hi-C matrix for given chromosome
    matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
    n_bins = matrix.shape[0]

    # Eigenvectors for given chromosome
    evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
    E1 = evec_chr['E1'].values
    E2 = evec_chr['E2'].values
    E3 = evec_chr['E3'].values

    # Gene density values for given chromosome
    gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
    gene_density = gene_density_chr["gene_count"].values

    # Repeat density values for given chromosome
    repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
    repeat_density = repeat_density_chr["repeat_count"].values
    
    # Plotting 
    ## Plot Hi-C and tracks
    fig, ax = plt.subplots(figsize=(10, 10))

    im = ax.matshow(
        matrix,
        norm=LogNorm(vmax=0.1), # log
        cmap='bwr'
    )
    ax.set_xlim(0, n_bins)
    ax.set_ylim(n_bins, 0)

    ## Colorbar showing corrected frequencies
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label('Corrected frequencies', fontsize=12)
    cbar.ax.tick_params(labelsize=10)

    
    ## Genomic x-axis in Mb
    genomic_ticks = np.arange(0, n_bins, step=10)
    genomic_labels = (genomic_ticks * resolution) / 1e6
    ax.set_yticks(genomic_ticks)
    ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
    ax.set_ylabel("Genomic Position (Mb)", fontsize=12)

    ax_prev = ax # Make the plots share x-axis

    # E1 Compartment track (bars)
    ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
    positions = np.arange(len(E1))
    ### Colors for A/B compartments
    colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")

    ax1.bar(positions, 
            E1, 
            width=1.0, 
            color=colors, 
            edgecolor="none")
    
    ax1.axhline(0, color='black', lw=0.5)
    ax1.set_ylabel('E1', fontsize=12)
    ax1.set_xticks([])
    ax_prev = ax1

    ## Compartment boundaries (E1 sign changes)
    boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]
    for b in boundaries:
        ax.axhline(b, color='k', lw=0.5)
        ax.axvline(b, color='k', lw=0.5)

    ## E2 track
    ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    positions_E2 = np.arange(len(E2))
    ### Colors for A/B compartments
    colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")

    ax2.bar(positions_E2, 
            E2, 
            width=1.0, 
            color=colors_E2, 
            edgecolor="none")
    
    ax2.axhline(0, color='black', lw=0.5)
    ax2.set_ylabel('E2', fontsize=12)
    ax2.set_xticks([])
    ax_prev = ax2

    ## E3 track
    ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    positions_E3 = np.arange(len(E3))
    ### Colors for A/B compartments
    colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
    ax3.bar(positions_E3, 
            E3, 
            width=1.0, 
            color=colors_E3, 
            edgecolor="none")
    
    ax3.axhline(0, color='black', lw=0.5)
    ax3.set_ylabel('E3', fontsize=12)
    ax3.set_xticks([])
    ax_prev = ax3

    ## Gene density line plot
    ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
    ax_gene.set_ylabel("Gene\nDensity", fontsize=12)
    ax_gene.set_xticks([])
    ax_gene.set_ylim(0, max(gene_density)*1.1)
    ax_prev = ax_gene

    ## Repeat density line plot
    ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
    ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
    ax_repeat.set_ylabel("Repeat\nDensity", fontsize=12)
    ax_repeat.set_xticks([])
    ax_repeat.set_ylim(0, max(repeat_density)*1.1)


    # Title for plot
    fig.text(
        0.5, 0.91,
        f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
        ha="center",
        va="top",
        fontsize=20,
        color='#0E7C7BFF',
        fontname='Times New Roman'
    )

    plt.show()


# Find centromere for remaining chromosomes

In [ ]:
# Largest chromosome
chrom = 'subgenom1_SUPER_2'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
# Found studying contact map
centromere_start = 3000000
centromere_end = 3400000
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.legend(['Centromere'], loc='upper right')

ax_prev = ax # Make the plots share x-axis

## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=8)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=2)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=15)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#0E7C7BFF',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
# Largest chromosome
chrom = 'subgenom2_SUPER_1'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
# Found studying contact map
centromere_start = 5000000
centromere_end = 5200000
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.legend(['Centromere'], loc='upper right')

ax_prev = ax # Make the plots share x-axis

## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=8)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=2)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=15)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#0E7C7BFF',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
# Largest chromosome
chrom = 'subgenom2_SUPER_7'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
# Same as SUPER 7 in subgenome 1
centromere_start = 2461517
centromere_end = 2920085
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.legend(['Centromere'], loc='upper right')

ax_prev = ax # Make the plots share x-axis

## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=8)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=2)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=15)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#0E7C7BFF',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
# Largest chromosome
# Found studying contact map
chrom = 'subgenom2_SUPER_8'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
# Found studying contact map
centromere_start = 950000
centromere_end = 1200000
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.legend(['Centromere'], loc='upper right')

ax_prev = ax # Make the plots share x-axis

## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=8)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=2)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=15)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#0E7C7BFF',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
# Largest chromosome
chrom = 'subgenom2_SUPER_9'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
# Found studying contact map
centromere_start = 1400000
centromere_end = 1800000
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.legend(['Centromere'], loc='upper right')

ax_prev = ax # Make the plots share x-axis

## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=8)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=2)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=15)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='red', alpha=0.8)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='purple', alpha=0.8)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='#0E7C7BFF',
    fontname='Times New Roman'
)

plt.show()

# Plot for results

## Find largest chromosome with predicted centromere 

In [ ]:
# Sort size in descending order
chromsizes = pd.DataFrame({
    "chrom": AmyRoux_file.chromnames,
    "size": AmyRoux_file.chromsizes
})
chromsize = chromsizes.sort_values(by='size', ascending=False)

print(chromsize)

In [ ]:
# Largest chromosome with annotated centromere
chrom = 'subgenom2_SUPER_2'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
centromere_start, centromere_end = centromeres[chrom]
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,10))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)', fontsize=14)
cbar.ax.tick_params(labelsize=10)

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)", fontsize=14)

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
#ax.legend(['Centromere'], loc='upper right')


## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', fontsize=14, labelpad=10)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', fontsize=14, labelpad=5)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.15, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', fontsize=14, labelpad=5)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='#4DAF4A', alpha=0.9)
ax_gene.set_ylabel("Gene\nDensity", fontsize=14,labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.15, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='#3A86C8', alpha=0.9)
ax_repeat.set_ylabel("Repeat\nDensity", fontsize=14)
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    fontsize=20,
    color='black',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
# Set fontsizes for result plots (to be the same across species)
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 30,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

In [ ]:
# Largest chromosome with annotated centromere
chrom = 'subgenom2_SUPER_2'
chromosome_number = chrom.split('_')[-1]
print(chromosome_number)

# Centromere coordinates
centromere_start, centromere_end = centromeres[chrom]
# Convert centromere coordinates to bins
centromere_start_bin = centromere_start // resolution
centromere_end_bin = centromere_end // resolution


# Hi-C matrix for given chromosome
matrix = AmyRoux_file.matrix(balance=True).fetch(chrom)
n_bins = matrix.shape[0]



# Eigenvectors for given chromosome
evec_chr = eigenvector_track_all.query(f'chrom == "{chrom}"')
E1 = evec_chr['E1'].values
E2 = evec_chr['E2'].values
E3 = evec_chr['E3'].values

# Gene density values for given chromosome
gene_density_chr = gene_density_AmyRoux[gene_density_AmyRoux["chrom"] == chrom]
gene_density = gene_density_chr["gene_count"].values


# Repeat density values for given chromosome
repeat_density_chr = repeat_density_AmyRoux[repeat_density_AmyRoux["chrom"] == chrom]
repeat_density = repeat_density_chr["repeat_count"].values

# Plotting
## Plot Hi-C
fig, ax = plt.subplots(figsize=(10,12))

im = ax.matshow(
    matrix,
    norm=LogNorm(vmax=0.1), # log
    cmap='bwr'
)

ax.set_xlim(0, n_bins)
ax.set_ylim(n_bins, 0)

## Colorbar showing corrected frequencies
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Corrected frequencies (log)')
cbar.ax.tick_params()

## Genomic x-axis in Mb
genomic_ticks = np.arange(0, n_bins, step=10)
genomic_labels = (genomic_ticks * resolution) / 1e6

ax.set_yticks(genomic_ticks)
ax.set_yticklabels([f"{x:.2f}" for x in genomic_labels])
ax.set_ylabel("Genomic Position (Mb)")

## Predicted centromere shown as dashed lines in green in Hi-C matrix
ax.axhline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_start_bin, color='black', linestyle='--', linewidth=1)
ax.axhline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
ax.axvline(centromere_end_bin, color='black', linestyle='--', linewidth=1)
#ax.legend(['Centromere'], loc='upper right')


## E1 Compartment track (bars)
ax1 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax)
positions = np.arange(len(E1))
### Colors A/B compartments
colors = np.where(E1 >= 0, "#D60F0F", "#034AA6FF")
ax1.bar(
    positions,
    E1,
    width=1.0,
    color=colors,
    edgecolor="none"
)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('E1', labelpad=10)
ax1.set_xticks([])
ax_prev = ax1

## Compartment boundaries using E1 shifts
boundaries = np.where(np.diff((E1 > 0).astype(int)))[0]

for b in boundaries:
    ax.axhline(b, color='k', lw=0.5)
    ax.axvline(b, color='k', lw=0.5)


## E2 track
ax2 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax1)
positions_E2 = np.arange(len(E2))
### Colors A/B compartments
colors_E2 = np.where(E2 >= 0, "#D60F0F", "#034AA6FF")
ax2.bar(
    positions_E2,
    E2,
    width=1.0,
    color=colors_E2,
    edgecolor="none"
)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('E2', labelpad=5)
ax2.set_xticks([])
ax_prev = ax2

## E3 track
ax3 = divider.append_axes("top", size="15%", pad=0.25, sharex=ax1)
positions_E3 = np.arange(len(E3))
### Colors A/B compartments
colors_E3 = np.where(E3 >= 0, "#D60F0F", "#034AA6FF")
ax3.bar(
    positions_E3,
    E3,
    width=1.0,
    color=colors_E3,
    edgecolor="none"
)
ax3.axhline(0, color='black', lw=0.5)
ax3.set_ylabel('E3', labelpad=5)
ax3.set_xticks([])
ax_prev = ax3

## Gene density line plot
ax_gene = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
ax_gene.plot(gene_density, lw=1, color='black', alpha=0.9)
ax_gene.set_ylabel("Gene\nDensity",labelpad=10)
ax_gene.set_xticks([])
ax_gene.set_ylim(0, max(gene_density)*1.1)
ax_prev = ax_gene

## Repeat density line plot
ax_repeat = divider.append_axes("top", size="15%", pad=0.25, sharex=ax_prev)
ax_repeat.plot(repeat_density, lw=1, color='black', alpha=0.9)
ax_repeat.set_ylabel("Repeat\nDensity")
ax_repeat.set_xticks([])
ax_repeat.set_ylim(0, max(repeat_density)*1.1)

# Align all y-axis labels
for a in [ax1, ax2, ax3, ax_gene, ax_repeat]:
    a.yaxis.set_label_coords(-0.11, 0.5)


# Title
fig.text(
    0.5,
    0.91,
    f"Chromosome {chromosome_number} – A. rouxii ({resolution//1000} kb)",
    ha="center",
    va="top",
    color='#5FC0BF',
    fontsize=25,
    fontname='Times New Roman'
)

plt.show()